# 11.03 -- Flash Attention: Memory-Efficient Attention from First Principles

**Module:** 11 -- LLM Systems Engineering  
**Notebook:** 3 of 5  
**Prerequisites:** 09.01 (KV Cache), 11.01 (GPU Memory Planning)

---

## Learning Objectives

1. Understand why standard attention is memory-bandwidth bound, not compute-bound
2. Derive the IO complexity of naive attention vs Flash Attention
3. Implement the Flash Attention tiling algorithm from scratch
4. Implement the online softmax trick that makes tiling exact (not approximate)
5. Measure memory usage and throughput gains on real hardware

---

## 1. The Problem with Standard Attention

Standard multi-head attention computes:
```
  S = Q K^T / sqrt(d)     materialised in HBM: O(N^2) memory
  P = softmax(S)           materialised in HBM: O(N^2) memory
  O = P V                  result: O(N * d)
```

For a sequence of N=4096 tokens, one attention head with d=128 produces
an S matrix of shape (4096, 4096) = 16.7 M float16 values = 33 MB.
With 32 heads and 32 layers, that is 33 * 32 * 32 = 33 GB of HBM bandwidth
just for the attention score matrices -- per forward pass.

The core issue is not compute (FLOP count); it is HBM reads and writes:

```
  Standard attention IO:     O(N^2 * d)  reads + writes  (quadratic in seq len)
  Flash Attention IO:        O(N * d^2 / M)               where M = SRAM size
                          ~  O(N * d)   in practice       (linear in seq len)
```

Flash Attention achieves this by never materialising the full N x N score matrix
in HBM. Instead it tiles the computation into blocks that fit in on-chip SRAM.

---

## 2. GPU Memory Hierarchy

```
  HBM (High Bandwidth Memory):  80 GB, ~2 TB/s bandwidth  <- where tensors live
  L2 cache:                     40 MB, ~10 TB/s            <- automatic, not programmed
  SRAM (shared memory):         192 KB per SM, ~20 TB/s    <- manually managed in CUDA
  Registers:                    256 KB per SM               <- fastest, most limited

  Key: SRAM is 10x faster than HBM but 400,000x smaller.
  Tiling keeps hot data in SRAM, dramatically reducing HBM traffic.
```


In [ ]:
# Environment setup.
#
# We implement the Flash Attention algorithm in pure PyTorch.
# This is not CUDA-optimised (the real speedup requires custom CUDA kernels
# with explicit shared memory management), but it demonstrates the
# tiling logic, the online softmax update, and the IO complexity reduction.
# We then benchmark torch.nn.functional.scaled_dot_product_attention
# (which uses FlashAttention-2 under the hood on CUDA) vs naive attention.

import torch
import torch.nn.functional as F
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import math
import time
import warnings
warnings.filterwarnings('ignore')

torch.manual_seed(42)
np.random.seed(42)

device    = 'cuda' if torch.cuda.is_available() else 'cpu'
HAS_CUDA  = device == 'cuda'
print(f'Device: {device}')
if HAS_CUDA:
    props = torch.cuda.get_device_properties(0)
    total_sram_kb = props.max_shared_memory_per_block / 1024
    print(f'GPU: {props.name}')
    print(f'  HBM total:         {props.total_memory / 1e9:.1f} GB')
    print(f'  Max SRAM per block: {total_sram_kb:.0f} KB')
    print(f'  SMs:               {props.multi_processor_count}')


## 3. Online Softmax: The Key Algorithmic Insight


In [ ]:
# The online softmax algorithm.
#
# Standard softmax requires two passes over the data:
#   Pass 1: find the maximum value (for numerical stability)
#   Pass 2: compute exp(x - max) and divide by the sum
#
# This means we must read the entire row before we can write any output.
# For Flash Attention, we need to compute softmax across a row that is
# split into tiles -- we cannot wait until we have seen all tiles.
#
# Online softmax (Milakov and Gimelshein, 2018) maintains running statistics
# that can be updated incrementally as new tiles arrive:
#
#   m_i  = running maximum of scores seen so far
#   l_i  = running sum of exp(scores - m_i)
#
# When a new tile arrives with maximum m_new and sum l_new:
#   m_combined = max(m_i, m_new)
#   l_combined = exp(m_i - m_combined) * l_i + exp(m_new - m_combined) * l_new
#
# The output O can be updated similarly:
#   O_combined = (exp(m_i - m_combined) * l_i * O_i
#                 + exp(m_new - m_combined) * P_new * V_new) / l_combined
#
# This allows single-pass attention computation over arbitrary sequence lengths
# without ever materialising the full score matrix.

def online_softmax_update(
    m_prev: torch.Tensor,   # (B, H) running max
    l_prev: torch.Tensor,   # (B, H) running normaliser
    O_prev: torch.Tensor,   # (B, H, d) running output accumulator
    scores_new: torch.Tensor,  # (B, H, block_size) new tile of scores
    V_new:      torch.Tensor,  # (B, H, block_size, d) corresponding V tile
) -> tuple:
    """
    Update the running softmax statistics with a new tile.

    Returns updated (m, l, O) that are mathematically equivalent to
    having computed softmax over all scores seen so far.
    """
    # Max and sum over new tile
    m_new = scores_new.max(dim=-1).values   # (B, H)
    p_new = torch.exp(scores_new - m_new.unsqueeze(-1))  # (B, H, block)
    l_new = p_new.sum(dim=-1)               # (B, H)

    # Combine running stats with new tile
    m_combined = torch.maximum(m_prev, m_new)   # (B, H)

    # Rescale both contributions to the combined maximum
    alpha_prev = torch.exp(m_prev - m_combined)   # (B, H)
    alpha_new  = torch.exp(m_new  - m_combined)   # (B, H)

    l_combined = alpha_prev * l_prev + alpha_new * l_new   # (B, H)

    # Update output accumulator
    # O_prev: (B, H, d);  p_new @ V_new: (B, H, d)
    O_new_contrib = torch.bmm(
        p_new.view(-1, p_new.shape[-1], 1).squeeze(-1).unsqueeze(1),
        V_new.view(-1, V_new.shape[-2], V_new.shape[-1])
    )  # simplified; see tiled implementation below

    # Rescale previous output and add new contribution
    O_combined = (
        alpha_prev.unsqueeze(-1) * l_prev.unsqueeze(-1) * O_prev
        + alpha_new.unsqueeze(-1) * (p_new @ V_new.squeeze(1) if V_new.dim() == 4
                                     else p_new @ V_new)
    ) / l_combined.unsqueeze(-1)

    return m_combined, l_combined, O_combined


# Verify online softmax gives the same result as standard softmax
torch.manual_seed(7)
B, H, N, d = 2, 4, 64, 32
Q = torch.randn(B, H, N, d)
K = torch.randn(B, H, N, d)
V = torch.randn(B, H, N, d)
scale = d ** -0.5

# Standard attention (reference)
S_full = torch.matmul(Q, K.transpose(-2, -1)) * scale  # (B, H, N, N)
P_full = torch.softmax(S_full, dim=-1)
O_ref  = torch.matmul(P_full, V)  # (B, H, N, d)

# Verify softmax normalisation
row_sums = P_full.sum(dim=-1)
print(f'Standard softmax row sum: mean={row_sums.mean():.6f}, std={row_sums.std():.2e}')
print('(Each row should sum to exactly 1.0)')


In [ ]:
# Flash Attention tiled forward pass (pure PyTorch reference implementation).
#
# This implements the core Flash Attention algorithm from Dao et al. (2022).
# It computes exact attention (not approximate) by processing the score
# matrix in tiles of size (BLOCK_Q, BLOCK_K) and using online softmax
# to accumulate the output without materialising the full N x N matrix.
#
# Algorithm (simplified, causal mask omitted for clarity):
#
#   Initialise: O = 0, l = 0, m = -inf   (all shape: B, H, N, d / B, H, N / B, H, N)
#
#   For each K/V block (j = 0, 1, ..., N/Bc - 1):
#     Load K_j, V_j from HBM to SRAM
#     For each Q block (i = 0, 1, ..., N/Br - 1):
#       Load Q_i from HBM to SRAM
#       Compute S_ij = Q_i K_j^T / sqrt(d)   (stays in SRAM)
#       Run online softmax update to get new m_i, l_i, O_i
#       Write O_i back to HBM
#
# HBM IO count:
#   Read Q, K, V:         3 * N * d   (each read once from HBM)
#   Write O:              N * d
#   Total:                4 * N * d   <- LINEAR in N
#
# Standard attention HBM IO:
#   Read Q, K, V:         3 * N * d
#   Write S:              N^2
#   Read S for softmax:   N^2
#   Write P:              N^2
#   Read P, V for O:      N^2 + N * d
#   Write O:              N * d
#   Total:                ~4 * N^2   <- QUADRATIC in N

def flash_attention_forward(
    Q: torch.Tensor,  # (B, H, N, d)
    K: torch.Tensor,
    V: torch.Tensor,
    block_size: int = 32,
    causal:     bool = True,
) -> torch.Tensor:
    """
    Flash Attention forward pass: exact attention without materialising
    the full N x N score matrix.

    This Python implementation demonstrates the algorithm but runs on CPU/GPU
    using standard PyTorch ops. The speedup in production comes from a
    custom CUDA kernel that manually manages shared memory.
    """
    B, H, N, d = Q.shape
    scale       = d ** -0.5

    # Flatten batch and head dimensions for simplicity: (B*H, N, d)
    Q_flat = Q.view(B * H, N, d)
    K_flat = K.view(B * H, N, d)
    V_flat = V.view(B * H, N, d)
    BH = B * H

    # Running statistics
    O = torch.zeros(BH, N, d, dtype=Q.dtype, device=Q.device)
    l = torch.zeros(BH, N,    dtype=Q.dtype, device=Q.device)   # normaliser
    m = torch.full((BH, N),   float('-inf'), dtype=Q.dtype, device=Q.device)  # max

    n_blocks = math.ceil(N / block_size)

    for j in range(n_blocks):
        # Load K, V tile j  (simulates reading from HBM to SRAM)
        j_start = j * block_size
        j_end   = min(j_start + block_size, N)
        K_j = K_flat[:, j_start:j_end, :]  # (BH, bs, d)
        V_j = V_flat[:, j_start:j_end, :]  # (BH, bs, d)

        for i in range(n_blocks):
            # Load Q tile i
            i_start = i * block_size
            i_end   = min(i_start + block_size, N)
            Q_i = Q_flat[:, i_start:i_end, :]  # (BH, bs, d)

            # Causal mask: query position i should not attend to key position j > i
            if causal and j_start > i_end - 1:
                continue   # entire K_j block is in the future for all Q_i positions

            # Score tile: (BH, bs_q, bs_k)
            S_ij = torch.bmm(Q_i, K_j.transpose(1, 2)) * scale

            if causal:
                # Mask positions where key index > query index
                qi_range = torch.arange(i_start, i_end, device=Q.device).unsqueeze(1)
                kj_range = torch.arange(j_start, j_end, device=Q.device).unsqueeze(0)
                causal_mask = kj_range > qi_range   # True = should be masked
                S_ij = S_ij.masked_fill(
                    causal_mask.unsqueeze(0).expand(BH, -1, -1), float('-inf')
                )

            # Online softmax update for Q_i positions
            m_new = S_ij.max(dim=-1).values                          # (BH, bs_q)
            P_ij  = torch.exp(S_ij - m_new.unsqueeze(-1))            # (BH, bs_q, bs_k)
            l_new = P_ij.sum(dim=-1)                                  # (BH, bs_q)

            # Rescale existing running stats with new max
            m_i     = m[:, i_start:i_end]                             # (BH, bs_q)
            l_i     = l[:, i_start:i_end]
            O_i     = O[:, i_start:i_end, :]                          # (BH, bs_q, d)

            m_comb  = torch.maximum(m_i, m_new)
            alpha_i = torch.exp(m_i   - m_comb)                       # (BH, bs_q)
            alpha_j = torch.exp(m_new - m_comb)

            l_comb  = alpha_i * l_i + alpha_j * l_new

            # Update output accumulator
            O_new_contrib = torch.bmm(P_ij, V_j)                      # (BH, bs_q, d)
            O[:, i_start:i_end, :] = (
                alpha_i.unsqueeze(-1) * l_i.unsqueeze(-1) * O_i
                + alpha_j.unsqueeze(-1) * O_new_contrib
            ) / l_comb.unsqueeze(-1)

            m[:, i_start:i_end] = m_comb
            l[:, i_start:i_end] = l_comb

    return O.view(B, H, N, d)


# Correctness check: flash attention vs standard attention
torch.manual_seed(99)
B, H, N, d = 1, 2, 64, 32
Q = torch.randn(B, H, N, d)
K = torch.randn(B, H, N, d)
V = torch.randn(B, H, N, d)

# Reference: standard attention with causal mask
scale  = d ** -0.5
S      = torch.matmul(Q, K.transpose(-2, -1)) * scale
mask   = torch.tril(torch.ones(N, N)).bool()
S      = S.masked_fill(~mask.unsqueeze(0).unsqueeze(0), float('-inf'))
P_ref  = torch.softmax(S, dim=-1)
O_ref  = torch.matmul(P_ref, V)

# Flash attention
O_flash = flash_attention_forward(Q, K, V, block_size=16, causal=True)

max_err = (O_flash - O_ref).abs().max().item()
mean_err = (O_flash - O_ref).abs().mean().item()
print(f'Flash Attention correctness check (causal=True):')
print(f'  Max absolute error:  {max_err:.2e}')
print(f'  Mean absolute error: {mean_err:.2e}')
print(f'  Verified (max_err < 1e-4): {max_err < 1e-4}')


In [ ]:
# Memory and throughput analysis: standard vs Flash Attention.
#
# We measure two quantities:
#
#   1. Peak memory: torch.cuda.max_memory_allocated() during the forward pass.
#      Standard attention allocates an N x N score matrix; Flash Attention does not.
#      On CPU we use psutil to estimate memory; the IO analysis is analytical.
#
#   2. Throughput (tokens/s): measured by timing the full forward pass.
#      On CUDA, torch.nn.functional.scaled_dot_product_attention automatically
#      uses Flash Attention-2 when available. We compare this against our
#      naive reference implementation.
#
# The theoretical IO analysis shows:
#   Naive:  O(N^2)  HBM reads/writes  (N^2 from materialising S and P)
#   Flash:  O(N)    HBM reads/writes  (only read/write Q, K, V, O once)
# Crossover point: Flash Attention becomes faster when N^2 >> N * d,
# i.e., when N >> d. For d=128 the crossover is around N=128-256.

def naive_attention(Q, K, V, causal=True):
    """Standard attention with explicit score matrix materialisation."""
    scale = Q.shape[-1] ** -0.5
    S = torch.matmul(Q, K.transpose(-2, -1)) * scale
    if causal:
        N = Q.shape[-2]
        mask = torch.tril(torch.ones(N, N, device=Q.device)).bool()
        S = S.masked_fill(~mask.unsqueeze(0).unsqueeze(0), float('-inf'))
    P = torch.softmax(S, dim=-1)
    return torch.matmul(P, V)


import time

def timed_attention(fn, Q, K, V, n_runs=10, n_warmup=3):
    """Time an attention function with warmup and CUDA synchronisation."""
    for _ in range(n_warmup):
        _ = fn(Q, K, V)
        if HAS_CUDA: torch.cuda.synchronize()

    if HAS_CUDA:
        torch.cuda.reset_peak_memory_stats()

    latencies = []
    for _ in range(n_runs):
        if HAS_CUDA: torch.cuda.synchronize()
        t0 = time.perf_counter()
        out = fn(Q, K, V)
        if HAS_CUDA: torch.cuda.synchronize()
        latencies.append((time.perf_counter() - t0) * 1000)

    peak_mem = torch.cuda.max_memory_allocated() / 1024**2 if HAS_CUDA else None
    return np.mean(latencies), np.std(latencies), peak_mem


SEQ_LENS = [128, 256, 512, 1024, 2048]
if HAS_CUDA:
    SEQ_LENS = [128, 256, 512, 1024, 2048, 4096]

B, H, d = 1, 8, 64

results_naive = []
results_flash = []

# Analytical HBM IO estimates
def io_naive(N, d):  return 4 * N * N * 2 + 3 * N * d * 2   # bytes (float16)
def io_flash(N, d):  return 4 * N * d * 2                    # bytes (float16)

print(f'Attention benchmark (B={B}, H={H}, d={d}):')
print(f'{"N":>6}  {"Naive ms":>10}  {"Flash ms":>10}  '
      f'{"Speedup":>8}  {"IO naive MB":>12}  {"IO flash MB":>12}')
print('-' * 70)

for N in SEQ_LENS:
    Q = torch.randn(B, H, N, d, device=device, dtype=torch.float16)
    K = torch.randn(B, H, N, d, device=device, dtype=torch.float16)
    V = torch.randn(B, H, N, d, device=device, dtype=torch.float16)

    # Naive attention
    def naive_fn(q, k, v): return naive_attention(q.float(), k.float(), v.float())
    t_naive, _, _ = timed_attention(naive_fn, Q, K, V, n_runs=5, n_warmup=2)

    # Flash attention (PyTorch built-in, uses FlashAttention-2 on CUDA)
    def flash_fn(q, k, v):
        return F.scaled_dot_product_attention(q, k, v, is_causal=True)
    t_flash, _, peak_mem = timed_attention(flash_fn, Q, K, V, n_runs=5, n_warmup=2)

    speedup   = t_naive / max(t_flash, 1e-9)
    io_n_mb   = io_naive(N, d) / 1024**2
    io_f_mb   = io_flash(N, d) / 1024**2

    results_naive.append((N, t_naive))
    results_flash.append((N, t_flash))

    mem_str = f'{peak_mem:.1f} MB' if peak_mem else 'N/A'
    print(f'{N:>6}  {t_naive:>10.2f}  {t_flash:>10.2f}  '
          f'{speedup:>8.1f}x  {io_n_mb:>12.1f}  {io_f_mb:>12.1f}')


In [ ]:
# Visualise the Flash Attention analysis.
#
# Four panels:
#   1. Tiling diagram: how blocks map to the N x N score matrix
#   2. IO complexity comparison: N^2 vs N scaling
#   3. Latency: naive vs Flash Attention across sequence lengths
#   4. Memory vs N: the quadratic score matrix dominates for naive attention

fig = plt.figure(figsize=(15, 10))
gs  = gridspec.GridSpec(2, 2, figure=fig, hspace=0.42, wspace=0.32)

# Panel 1: tiling diagram
ax = fig.add_subplot(gs[0, 0])
N_DIAG, BLOCK = 8, 2
n_blocks = N_DIAG // BLOCK

# Draw the full N x N matrix
for i in range(N_DIAG):
    for j in range(N_DIAG):
        color = '#dce9f5' if j <= i else '#f5dcdc'
        ax.add_patch(plt.Rectangle((j, N_DIAG-1-i), 1, 1,
                                   color=color, ec='gray', lw=0.5))

# Highlight one active tile pair
bi, bj = 1, 0  # Q-block 1, K-block 0
for qi in range(bi*BLOCK, (bi+1)*BLOCK):
    for kj in range(bj*BLOCK, (bj+1)*BLOCK):
        if kj <= qi:
            ax.add_patch(plt.Rectangle((kj, N_DIAG-1-qi), 1, 1,
                                       color='#2ca02c', ec='black', lw=1.5, alpha=0.7))

ax.set_xlim(0, N_DIAG); ax.set_ylim(0, N_DIAG)
ax.set_xticks(range(N_DIAG)); ax.set_yticks(range(N_DIAG))
ax.set_xticklabels([f'K{j}' for j in range(N_DIAG)], fontsize=7)
ax.set_yticklabels([f'Q{N_DIAG-1-i}' for i in range(N_DIAG)], fontsize=7)
ax.set_title(f'Flash Attention Tiling\n'
             f'Green = active tile (Q-block {bi}, K-block {bj})\n'
             f'Blue = visible (causal), Red = masked', fontsize=10)

from matplotlib.patches import Patch
ax.legend(handles=[
    Patch(color='#dce9f5', label='Allowed (causal)'),
    Patch(color='#f5dcdc', label='Masked (future)'),
    Patch(color='#2ca02c', alpha=0.7, label='Active tile'),
], fontsize=7, loc='lower right')

# Panel 2: IO complexity
ax = fig.add_subplot(gs[0, 1])
Ns      = np.arange(128, 8193, 64)
d_val   = 128
io_n_mb_arr = [io_naive(n, d_val) / 1024**2 for n in Ns]
io_f_mb_arr = [io_flash(n, d_val) / 1024**2 for n in Ns]
ax.plot(Ns, io_n_mb_arr, lw=2.5, color='#d62728', label='Naive O(N^2)')
ax.plot(Ns, io_f_mb_arr, lw=2.5, color='#2ca02c', label='Flash O(N)')
ax.axvline(d_val, color='gray', ls=':', alpha=0.6)
ax.text(d_val+50, max(io_f_mb_arr)*0.85, f'N=d={d_val}\ncrossover', fontsize=8, color='gray')
ax.set_xlabel('Sequence length N', fontsize=11)
ax.set_ylabel('HBM IO (MB, float16, d=128)', fontsize=11)
ax.set_title('HBM IO Complexity\nFlash Attention eliminates O(N^2) reads/writes', fontsize=12)
ax.legend(fontsize=9)
ax.grid(alpha=0.3)

# Panel 3: measured latency
ax = fig.add_subplot(gs[1, 0])
ns_naive, ts_naive = zip(*results_naive)
ns_flash, ts_flash = zip(*results_flash)
ax.plot(ns_naive, ts_naive, 'o-', lw=2.5, color='#d62728', ms=7, label='Naive attention')
ax.plot(ns_flash, ts_flash, 's-', lw=2.5, color='#2ca02c', ms=7, label='Flash Attention (torch SDPA)')
ax.set_xlabel('Sequence length N', fontsize=11)
ax.set_ylabel('Latency (ms)', fontsize=11)
ax.set_title(f'Measured Latency: Naive vs Flash\n(B={B}, H={H}, d={d})', fontsize=12)
ax.legend(fontsize=9)
ax.grid(alpha=0.3)

# Panel 4: memory usage vs N
ax = fig.add_subplot(gs[1, 1])
# Memory for score matrix S: N^2 * 4 bytes (float32) for naive
mem_naive_gb = [N**2 * H * 4 / 1024**3 for N in Ns]   # full S per head
mem_flash_gb = [N * d_val * H * 4 / 1024**3 for N in Ns]  # only tiles (simplified)
ax.plot(Ns, [m * 1024 for m in mem_naive_gb], lw=2.5, color='#d62728', label='Naive (score matrix, MB)')
ax.plot(Ns, [m * 1024 for m in mem_flash_gb], lw=2.5, color='#2ca02c', label='Flash (no score matrix, MB)')
ax.axhline(1024, color='#1f77b4', ls='--', lw=1.5, label='1 GB threshold')
ax.set_xlabel('Sequence length N', fontsize=11)
ax.set_ylabel('Memory for attention scores (MB)', fontsize=11)
ax.set_title('Score Matrix Memory\n8 heads, float32', fontsize=12)
ax.legend(fontsize=9)
ax.grid(alpha=0.3)

plt.suptitle('Flash Attention: Memory-Efficient Attention Analysis', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('../figures/11_flash_attention.png', dpi=150, bbox_inches='tight')
plt.show()

print()
speedups = [t_n/max(t_f,1e-9) for (_, t_n), (_, t_f) in zip(results_naive, results_flash)]
print(f'Mean speedup Flash vs Naive: {np.mean(speedups):.1f}x')
print(f'Speedup grows with N (more memory-bound at longer sequences).')


## 4. Flash Attention Variants

**Flash Attention-2** (Dao, 2023) improves on Flash Attention-1 with:
- Fewer non-matrix-multiply FLOPs (rescaling moved outside inner loop)
- Better thread-block parallelism (parallelise over sequence length, not just batch/heads)
- Causal masking only applied to boundary tiles (not all tiles)
- Result: ~2x faster than FA-1 on A100

**Flash Attention-3** (Shah et al., 2024) for H100:
- Exploits H100 WGMMA (Warpgroup Matrix Multiply) instructions
- Overlaps async memory copies with computation
- ~2x faster than FA-2 on H100

**Using Flash Attention in PyTorch**

```python
import torch
import torch.nn.functional as F

# torch >= 2.0: SDPA automatically uses FA-2 on CUDA if available
output = F.scaled_dot_product_attention(
    query, key, value,
    attn_mask=None,
    dropout_p=0.0,
    is_causal=True,
)

# Explicitly request Flash Attention backend:
with torch.backends.cuda.sdp_kernel(
    enable_flash=True,
    enable_math=False,
    enable_mem_efficient=False,
):
    output = F.scaled_dot_product_attention(query, key, value, is_causal=True)
```

## 5. Exercises

1. **Block size sensitivity**: Run `flash_attention_forward` with block sizes
   [8, 16, 32, 64]. Verify correctness for all sizes and plot the wall-clock
   time vs block size. Explain the trade-off: larger blocks reduce loop overhead
   but require more SRAM.

2. **Causal vs non-causal memory**: Measure peak GPU memory for causal vs
   bidirectional Flash Attention (set `is_causal=False`) at N=4096.
   How does causal masking reduce memory? (Hint: roughly half the score tiles
   are fully masked and need not be computed or stored.)

3. **Numerical precision**: Compare the output of `flash_attention_forward`
   against naive attention in float32 vs float16. Where does the error come from?
   Measure max error as a function of sequence length.

4. **IO counting**: Instrument the `flash_attention_forward` function to count
   the actual number of tensor read/write operations to simulate HBM traffic.
   Verify that the count grows linearly with N (not quadratically).

5. **Integration**: Replace `nn.MultiheadAttention` in a GPT-2 forward pass
   with `F.scaled_dot_product_attention`. Verify identical outputs (to float16
   precision) and measure the memory reduction at sequence length 512.

---

## 6. References

- [Dao et al. (2022) -- FlashAttention: Fast and Memory-Efficient Exact Attention with IO-Awareness](https://arxiv.org/abs/2205.14135)
- [Dao (2023) -- FlashAttention-2: Faster Attention with Better Parallelism and Work Partitioning](https://arxiv.org/abs/2307.08691)
- [Shah et al. (2024) -- FlashAttention-3: Fast and Accurate Attention for H100 GPUs](https://arxiv.org/abs/2407.08608)
- [Milakov and Gimelshein (2018) -- Online normalizer calculation for softmax](https://arxiv.org/abs/1805.02867)
